# 01_dataset_understanding

## Imports

In [4]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

### Data Loading

In [5]:
data_listing = pd.read_csv("../data/CRMLSListingMaster.csv", low_memory=False)
data_sold = pd.read_csv("../data/CRMLSSoldMaster.csv", low_memory=False)

data_listing.head()

,source_file,file_period,sort_date,OriginalListPrice,ListingKey,ListAgentEmail,CloseDate,ClosePrice,ListAgentFirstName,ListAgentLastName,...,MainLevelBedrooms,NewConstructionYN,GarageSpaces,HighSchoolDistrict,PostalCode,BuyerOfficeName__dup2,AssociationFee,LotSizeSquareFeet,MiddleOrJuniorSchoolDistrict,UnparsedAddress__dup2
0,CRMLSListing202401.csv,202401,2024-01-01,1399990.0,1043760653,joe@porchlightsocal.com,2024-03-15,1270000.0,Joseph,Vahedi,...,2.0,NaN,2.0,Vista Unified,92083,eXp Realty of Southern CA,252.0,9897.0,NaN,123 Flores Lane
1,CRMLSListing202401.csv,202401,2024-01-01,789900.0,1053093534,kiva@kivakendrick.com,2024-02-26,800000.0,Kiva,Kendrick,...,NaN,False,2.0,Los Angeles Unified,90746,R & R Realty,NaN,5070.0,NaN,19922 Enslow Drive
2,CRMLSListing202401.csv,202401,2024-01-01,1050000.0,1054014526,sarah.pearce@cbrealty.com,NaN,NaN,Sarah,Pearce,...,NaN,False,NaN,NaN,92262,NaN,NaN,10019.0,NaN,2030 E Acacia Road
3,CRMLSListing202401.csv,202401,2024-01-01,1250000.0,1054022382,mricks@mypsrealtor.com,2024-02-26,1199000.0,Michael,Ricks,...,NaN,False,2.0,NaN,92262,BHG Desert Lifestyle Properties,NaN,10454.0,NaN,1695 E Buena Vista Drive
4,CRMLSListing202401.csv,202401,2024-01-01,4750000.0,1054022420,james@larealproperty.com,NaN,NaN,James,Maxwell,...,NaN,False,2.0,NaN,90291,NaN,NaN,3084.0,NaN,2915 S Grand Canal


Check shapes:

In [6]:
data_sold.shape, data_listing.shape

((397603, 87), (540183, 87))

## Missing Value Analysis Functions

In [7]:
def missing_summary(df, df_name="data"):
    missing_count = df.isna().sum()
    missing_pct = df.isna().mean() * 100

    summary = pd.DataFrame({
        "column": df.columns,
        "missing_count": missing_count.values,
        "missing_pct": missing_pct.values
    }).sort_values("missing_pct", ascending=False)

    high_missing = summary[summary["missing_pct"] > 90]

    print(f"\nMissing value summary for {df_name}:")
    display(summary)

    print(f"\nColumns in {df_name} with >90% missing values:")
    display(high_missing)

    return summary, high_missing

Run missing value analysis on both datasets (Sold and Listing):

In [8]:
listing_missing_summary, listing_high_missing = missing_summary(data_listing, "data_listing")
sold_missing_summary, sold_high_missing = missing_summary(data_sold, "data_sold")


Missing value summary for data_listing:


,column,missing_count,missing_pct
63,BusinessType,540183,100.0
67,CoveredSpaces,540183,100.0
85,MiddleOrJuniorSchoolDistrict,540183,100.0
53,TaxYear,540183,100.0
31,TaxAnnualAmount,540183,100.0
...,...,...,...
33,PropertyType__dup2,0,0.0
29,ListingKeyNumeric,0,0.0
34,MlsStatus,0,0.0
65,ListPrice__dup2,0,0.0



Columns in data_listing with >90% missing values:


,column,missing_count,missing_pct
63,BusinessType,540183,100.000000
67,CoveredSpaces,540183,100.000000
85,MiddleOrJuniorSchoolDistrict,540183,100.000000
53,TaxYear,540183,100.000000
31,TaxAnnualAmount,540183,100.000000
28,AboveGradeFinishedArea,540183,100.000000
26,FireplacesTotal,540183,100.000000
58,ElementarySchoolDistrict,540183,100.000000
62,BelowGradeFinishedArea,537154,99.439264
59,CoBuyerAgentFirstName,525755,97.329053



Missing value summary for data_sold:


,column,missing_count,missing_pct
63,BusinessType,397603,100.0
65,CoveredSpaces,397603,100.0
80,MiddleOrJuniorSchoolDistrict,397603,100.0
54,TaxYear,397603,100.0
33,FireplacesTotal,397603,100.0
...,...,...,...
36,ListingKeyNumeric,0,0.0
24,ListOfficeName,0,0.0
23,DaysOnMarket,0,0.0
39,CountyOrParish,0,0.0



Columns in data_sold with >90% missing values:


,column,missing_count,missing_pct
63,BusinessType,397603,100.000000
65,CoveredSpaces,397603,100.000000
80,MiddleOrJuniorSchoolDistrict,397603,100.000000
54,TaxYear,397603,100.000000
33,FireplacesTotal,397603,100.000000
38,TaxAnnualAmount,397603,100.000000
35,AboveGradeFinishedArea,397603,100.000000
58,ElementarySchoolDistrict,397603,100.000000
7,WaterfrontYN,397355,99.937626
62,BelowGradeFinishedArea,395313,99.424049


Define the core columns that should be retained:

In [9]:
core_fields = [
    "ListingKey", "ListingId", "MlsStatus",
    "ListPrice", "OriginalListPrice", "ClosePrice", "CloseDate", "DaysOnMarket",
    "PropertyType", "PropertySubType",
    "BedroomsTotal", "BathroomsTotalInteger",
    "LivingArea", "LotSizeSquareFeet", "YearBuilt",
    "UnparsedAddress", "City", "StateOrProvince", "PostalCode", "CountyOrParish",
    "Latitude", "Longitude"
]

## Decide drop or retain Function

Decide which columns to drop vs retain based on missing percentage and core field status:

In [10]:
def decide_drop_or_retain(summary_df, core_fields, threshold=90):
    summary_df = summary_df.copy()

    summary_df["is_core_field"] = summary_df["column"].isin(core_fields)

    summary_df["recommended_action"] = np.where(
        (summary_df["missing_pct"] > threshold) & (~summary_df["is_core_field"]),
        "drop",
        "retain"
    )

    summary_df["reason"] = np.where(
        (summary_df["missing_pct"] > threshold) & (~summary_df["is_core_field"]),
        f"missing > {threshold}% and not a core field",
        np.where(
            (summary_df["missing_pct"] > threshold) & (summary_df["is_core_field"]),
            f"missing > {threshold}% but retained because it is a core field",
            "retain"
        )
    )

    drop_cols = summary_df.loc[
        summary_df["recommended_action"] == "drop", "column"
    ].tolist()

    retain_cols = summary_df.loc[
        summary_df["recommended_action"] == "retain", "column"
    ].tolist()

    return summary_df, drop_cols, retain_cols

Apply the decision function to both datasets and summarize the results

In [11]:
listing_decision_summary, listing_drop_cols, listing_retain_cols = decide_drop_or_retain(
    listing_missing_summary, core_fields
)

sold_decision_summary, sold_drop_cols, sold_retain_cols = decide_drop_or_retain(
    sold_missing_summary, core_fields
)

print("Listing dataset decision summary:")
display(listing_decision_summary)

print("Columns recommended to drop in data_listing:")
display(pd.DataFrame({"drop_column": listing_drop_cols}))

print("Sold dataset decision summary:")
display(sold_decision_summary)

print("Columns recommended to drop in data_sold:")
display(pd.DataFrame({"drop_column": sold_drop_cols}))

Listing dataset decision summary:


,column,missing_count,missing_pct,is_core_field,recommended_action,reason
63,BusinessType,540183,100.0,False,drop,missing > 90% and not a core field
67,CoveredSpaces,540183,100.0,False,drop,missing > 90% and not a core field
85,MiddleOrJuniorSchoolDistrict,540183,100.0,False,drop,missing > 90% and not a core field
53,TaxYear,540183,100.0,False,drop,missing > 90% and not a core field
31,TaxAnnualAmount,540183,100.0,False,drop,missing > 90% and not a core field
...,...,...,...,...,...,...
33,PropertyType__dup2,0,0.0,False,retain,retain
29,ListingKeyNumeric,0,0.0,False,retain,retain
34,MlsStatus,0,0.0,True,retain,retain
65,ListPrice__dup2,0,0.0,False,retain,retain


Columns recommended to drop in data_listing:


,drop_column
0,BusinessType
1,CoveredSpaces
2,MiddleOrJuniorSchoolDistrict
3,TaxYear
4,TaxAnnualAmount
5,AboveGradeFinishedArea
6,FireplacesTotal
7,ElementarySchoolDistrict
8,BelowGradeFinishedArea
9,CoBuyerAgentFirstName


Sold dataset decision summary:


,column,missing_count,missing_pct,is_core_field,recommended_action,reason
63,BusinessType,397603,100.0,False,drop,missing > 90% and not a core field
65,CoveredSpaces,397603,100.0,False,drop,missing > 90% and not a core field
80,MiddleOrJuniorSchoolDistrict,397603,100.0,False,drop,missing > 90% and not a core field
54,TaxYear,397603,100.0,False,drop,missing > 90% and not a core field
33,FireplacesTotal,397603,100.0,False,drop,missing > 90% and not a core field
...,...,...,...,...,...,...
36,ListingKeyNumeric,0,0.0,False,retain,retain
24,ListOfficeName,0,0.0,False,retain,retain
23,DaysOnMarket,0,0.0,True,retain,retain
39,CountyOrParish,0,0.0,True,retain,retain


Columns recommended to drop in data_sold:


,drop_column
0,BusinessType
1,CoveredSpaces
2,MiddleOrJuniorSchoolDistrict
3,TaxYear
4,FireplacesTotal
5,TaxAnnualAmount
6,AboveGradeFinishedArea
7,ElementarySchoolDistrict
8,WaterfrontYN
9,BelowGradeFinishedArea


Finalize the cleaned datasets by dropping the identified columns

In [12]:
data_listing_clean = data_listing.drop(columns=listing_drop_cols)
data_sold_clean = data_sold.drop(columns=sold_drop_cols)

print("Original shape of data_listing:", data_listing.shape)
print("Cleaned shape of data_listing:", data_listing_clean.shape)

print("Original shape of data_sold:", data_sold.shape)
print("Cleaned shape of data_sold:", data_sold_clean.shape)

Original shape of data_listing: (540183, 87)
Cleaned shape of data_listing: (540183, 74)
Original shape of data_sold: (397603, 87)
Cleaned shape of data_sold: (397603, 70)
